In [ ]:
%load_ext cudf.pandas

In [ ]:
#
import numpy as np  # linear algebra
import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

In [ ]:
%%time
### cell 0 ###

# load & cleanup
file = "/home/dias-benchmarks/notebooks/kkhandekar/environmental-vs-ai-startups-india-eda/input/indian-startup-recognized-by-dpiit/Startup_Counts_Across_India.csv"
df = pd.read_csv(file)
factor = 3000
df = pd.concat([df] * factor)
df.info()

In [ ]:
%%time
### cell 1 ###

df.drop("S No.", axis=1, inplace=True)
df.dropna(inplace=True)
df.reset_index(inplace=True, drop=True)

# view
df.head()

In [ ]:
%%time
### cell 2 ###

# Industry sub-categories for environmental & AI startups
env = ["Agriculture", "Green Technology", "Renewable Energy", "Waste Management"]
ai = ["AI", "Robotics", "Computer Vision"]

# combined df - environmental & AI startups only
df_ea = df[df["Industry"].isin(env + ai)].reset_index(drop=True)

# vectorized mapping of MainIndustry
category_map = {**{cat: "ENV" for cat in env}, **{cat: "AI" for cat in ai}}
df_ea["MainIndustry"] = df_ea["Industry"].map(category_map)

# basic stats
total = len(df_ea)
counts = df_ea["MainIndustry"].value_counts()
print(
    f"A total of {total} startups were started in India between 2016 & 2022, out of which {counts['ENV']} are environmental related & {counts['AI']} are AI startups."
)